# OCR invoice local CPU bằng Vintern GGUF

Notebook này dùng model quantized `rootonchair/Vintern-1B-v3_5-GGUF-ext` qua `llama-cpp-python`, chạy CPU-only (`n_gpu_layers=0`) nên không cần card rời. Lần chạy đầu sẽ tải file `.gguf` từ Hugging Face.

In [ ]:
# Cài dependency local. Nếu máy đã cài rồi thì có thể bỏ qua cell này.
# --prefer-binary giúp Windows lấy wheel CPU dựng sẵn thay vì build llama-cpp-python từ source.
%pip install --prefer-binary -r requirements-gguf-local.txt

In [ ]:
from pathlib import Path
from IPython.display import display

from ocr_invoice_gguf_local import (
    INVOICE_PROMPT,
    load_input_images_with_paths,
    load_llm,
    run_vintern_gguf,
    parse_model_json,
    scan_invoice_file,
    save_results,
)

In [ ]:
# Q4_K_M cân bằng chất lượng/tốc độ. Đổi sang Q4_K_S hoặc IQ4_XS nếu muốn nhẹ hơn.
llm = load_llm(
    quant="Q4_K_M",
    n_ctx=8192,
    n_threads=None,
    verbose=False,
)

print("Đã tải model GGUF local CPU")

## Chạy OCR cho ảnh hoặc PDF

In [ ]:
# Có thể là ảnh: .jpg, .jpeg, .png, .webp, .tif, ...
# Hoặc PDF: .pdf. Nếu là PDF, notebook sẽ render từng trang thành ảnh trước.
INPUT_PATH = Path("D:/Invoice_Scanner/4f30c1ad-c130-47a0-b164-41ad070218f2.pdf")
INPUT_IMAGE_DIR = INPUT_PATH.with_name(f"{INPUT_PATH.stem}_images")

images, source_type, image_paths = load_input_images_with_paths(
    INPUT_PATH,
    pdf_dpi=200,
    save_pdf_images=INPUT_PATH.suffix.lower() == ".pdf",
    pdf_image_dir=INPUT_IMAGE_DIR,
)

print("Loại file:", source_type)
print("Số trang/ảnh:", len(images))
if any(image_paths):
    print("Thư mục ảnh đã render:", INPUT_IMAGE_DIR)
    print("Ảnh trang đầu:", image_paths[0])

display(images[0])

In [ ]:
results = scan_invoice_file(
    input_path=INPUT_PATH,
    llm=llm,
    pdf_dpi=200,
    save_pdf_images=INPUT_PATH.suffix.lower() == ".pdf",
    pdf_image_dir=INPUT_IMAGE_DIR,
    max_tokens=2048,
    max_image_side=1800,
)

results

In [ ]:
output_json = INPUT_PATH.with_suffix(".gguf.json")

save_results(
    results=results,
    output_path=output_json,
    include_raw_response=False,
)

print("Đã lưu:", output_json)